In [6]:
import yfinance as yf

sp = yf.download("^GSPC", start="1989-01-01", end="2026-01-01", auto_adjust=False)
sp.columns = sp.columns.get_level_values(0)  # deja una sola fila de encabezados
sp.to_csv("gspc_diario.csv")

print(sp.shape)
print(sp.head(3))
print(sp.tail(3))

[*********************100%***********************]  1 of 1 completed

(9319, 6)
Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
1989-01-03  275.309998  275.309998  277.720001  273.809998  277.720001   
1989-01-04  279.429993  279.429993  279.750000  275.309998  275.309998   
1989-01-05  280.010010  280.010010  281.510010  279.429993  279.429993   

Price          Volume  
Date                   
1989-01-03  128500000  
1989-01-04  149700000  
1989-01-05  174040000  
Price         Adj Close        Close         High          Low         Open  \
Date                                                                          
2025-12-29  6905.740234  6905.740234  6920.209961  6888.759766  6903.600098   
2025-12-30  6896.240234  6896.240234  6913.250000  6893.470215  6900.439941   
2025-12-31  6845.500000  6845.500000  6901.419922  6844.549805  6898.819824   

Price           Volume  
Date                    
2025-12-29  3541750000  
2025-12-30  3309930000  
202

In [ ]:
"""
Tarea 2 AFML - Regresion logistica de referencia
Objetivo: predecir caida10 (caida >=10% del S&P 500 en los proximos 3 meses)
"""
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from comun import (puntaje, elegir_umbral, imprimir_umbral, metricas, tabla_train_test,
                   matriz_confusion, medir_prediccion, n_combinaciones, guardar_puntajes)
import statsmodels.api as sm

# ---------------------------------------------------------------
# 1. Datos
# ---------------------------------------------------------------
df = pd.read_csv("base_tarea2_caidas_sp500.csv", index_col="fecha")
df = df.dropna(subset=["caida10"])  # quita los ultimos 3 meses (objetivo desconocido)

PREDICTORES = ["cape_lag3", "mom_12m", "vix_fin", "baa10y_fin", "t10y3m_fin"]
X = df[PREDICTORES]
y = df["caida10"].astype(int)

# ---------------------------------------------------------------
# 2. Separacion entrenamiento / prueba 
# ---------------------------------------------------------------
INICIO_PRUEBA = "2015-01"
HORIZONTE = 3

es_train = X.index < INICIO_PRUEBA
idx_train = X.index[es_train][:-HORIZONTE]
idx_test = X.index[~es_train]

X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]

print(f"Entrenamiento: {idx_train[0]} a {idx_train[-1]} ({len(idx_train)} meses, {y_train.sum()} caidas)")
print(f"Prueba:        {idx_test[0]} a {idx_test[-1]} ({len(idx_test)} meses, {y_test.sum()} caidas)\n")

# ---------------------------------------------------------------
# 3. Modelo: estandarizar + logistica
# ---------------------------------------------------------------
pipe = Pipeline([
    ("escalar", StandardScaler()),
    ("logit", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

# ---------------------------------------------------------------
# 4. Validacion (solo dentro del entrenamiento) para elegir C
# ---------------------------------------------------------------
cv = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)
grilla = {"logit__C": [0.001, 0.01, 0.1, 1, 10, 100]}

busqueda = GridSearchCV(pipe, grilla, cv=cv, scoring="roc_auc")
t0 = time.time()
busqueda.fit(X_train, y_train)
tiempo = time.time() - t0

print("AUC de validacion por valor de C (sensibilidad a hiperparametros):")
res = pd.DataFrame(busqueda.cv_results_)[["param_logit__C", "mean_test_score", "std_test_score"]]
print(res.round(3).to_string(index=False))
print(f"\nMejor C: {busqueda.best_params_['logit__C']}  |  tiempo de entrenamiento: {tiempo:.2f} s\n")

modelo = busqueda.best_estimator_

# ---------------------------------------------------------------
# 5. Umbral (elegido en validacion out-of-fold, SIN mirar la prueba) y metricas
#    Umbral comun de comun.py (por cuantil, maximiza F1 out-of-fold).
# ---------------------------------------------------------------
umbral, info = elegir_umbral(modelo, X_train, y_train, cv)
imprimir_umbral(umbral, info)

s_train = puntaje(modelo, X_train)
s_test = puntaje(modelo, X_test)
tabla = tabla_train_test(metricas(y_train, s_train, umbral),
                         metricas(y_test, s_test, umbral))
print("Desempeno de la logistica:")
print(tabla.round(3).to_string())

print("\nMatriz de confusion en prueba (filas = real, columnas = predicho):")
print(matriz_confusion(y_test, s_test, umbral))

t_pred = medir_prediccion(modelo, X_test)
n_comb = n_combinaciones(busqueda)
print(f"\nTiempo de prediccion sobre la prueba ({len(X_test)} meses): {t_pred * 1000:.1f} ms"
      f"  |  combinaciones evaluadas: {n_comb}")

# ---------------------------------------------------------------
# 6. Interpretacion: coeficientes estandarizados
#    Signo positivo = esa variable aumenta la probabilidad de caida.
#    Magnitud = efecto de subir 1 desviacion estandar la variable.
# ---------------------------------------------------------------
coef = pd.Series(modelo.named_steps["logit"].coef_[0], index=PREDICTORES)
print("\nCoeficientes (variables estandarizadas):")
print(coef.round(3).to_string())

# Version statsmodels (sin regularizacion ni pesos) para ver significancia,
# igual que la salida de regresion vista en clases
Xs = sm.add_constant(StandardScaler().fit(X_train).transform(X_train))
logit_sm = sm.Logit(y_train.values, pd.DataFrame(Xs, columns=["const"] + PREDICTORES)).fit(disp=0)
print("\nLogit con statsmodels (entrenamiento, variables estandarizadas):")
print(logit_sm.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3))

# Guardar resultados para la tabla comparativa final
tabla.to_csv("resultados/resultados_logistica.csv")
pd.DataFrame({"C": [busqueda.best_params_["logit__C"]], "AUC_validacion": [busqueda.best_score_],
              "umbral": [umbral], "percentil_umbral": [info["percentil_corte"]],
              "tiempo_s": [tiempo], "tiempo_prediccion_s": [t_pred],
              "n_combinaciones": [n_comb]}).to_csv("resultados/resultados_logistica_extra.csv", index=False)
guardar_puntajes("logistica", idx_test, y_test, s_test, umbral)


Entrenamiento: 1990-01 a 2014-09 (297 meses, 41 caidas)
Prueba:        2015-01 a 2025-09 (129 meses, 21 caidas)

AUC de validacion por valor de C (sensibilidad a hiperparametros):
param_logit__C  mean_test_score  std_test_score
         0.001            0.533           0.319
          0.01            0.528           0.323
           0.1            0.471           0.267
             1            0.454           0.249
            10            0.480           0.261
           100            0.469           0.266

Mejor C: 0.001  |  tiempo de entrenamiento: 0.16 s

Desempeno de la logistica:
           Entrenamiento  Prueba  Brecha (train - test)
AUC                0.773   0.631                  0.142
Precision          0.318   0.258                  0.060
Recall             0.683   0.381                  0.302
F1                 0.434   0.308                  0.126

Matriz de confusion en prueba (filas = real, columnas = predicho):
        Pred 0  Pred 1
Real 0      85      23
Real 1    

In [ ]:
# ---------------------------------------------------------------
# 7. Importancia por permutacion (igual que SVM, RF y XGB) + coeficientes
#    Para comparar la interpretabilidad de los 5 modelos con la misma medida.
#    (correr despues de la celda anterior)
# ---------------------------------------------------------------
from sklearn.inspection import permutation_importance

imp = permutation_importance(modelo, X_test, y_test, scoring="roc_auc",
                             n_repeats=30, random_state=0)
imp_df = pd.DataFrame({"Caida del AUC": imp.importances_mean,
                       "Desv. est.": imp.importances_std,
                       "Coef. sklearn (C elegido)": coef.values,
                       "Coef. statsmodels": logit_sm.params[PREDICTORES].values,
                       "p-valor statsmodels": logit_sm.pvalues[PREDICTORES].values},
                      index=PREDICTORES)
print("Importancia por permutacion (muestra de prueba) y coeficientes:")
print(imp_df.sort_values("Caida del AUC", ascending=False).round(3).to_string())
imp_df.to_csv("resultados/importancia_logistica.csv", index_label="variable")
